# T02 — Inspector로 도구 검증 (Tutorial)

> 🎓 **학생 친화 튜토리얼** — 브라우저 UI(MCP Inspector)로 LLM 없이도 MCP 서버 도구를 시각적으로 검증합니다.

## 학습 목표
1. `mcp dev` 명령으로 Inspector 한 줄 기동
2. Inspector UI 단계별 사용법 (Connect → Tools → Run)
3. 3개 도구(시간/덧셈/면적) 시각적 검증
4. 브라우저 막힘 시 JSON-RPC 직접 통신 대안
5. Claude Desktop / Claude Code 등록 절차 이해

## Prerequisites
- T01 완료 (`tutorial_server.py` 존재)
- 브라우저 (Chrome / Edge / Firefox)
- 약 25분 소요

> 📖 **강의노트 매핑**: `Week_07.md §1.5` (Server Inspector). Skilljar 원본은 `skilljar/S6_02_mcp_inspector.ipynb`.


## §0. 강의노트 매핑

| 본 튜토리얼 단계 | 강의노트 위치 | Skilljar 레슨 |
|------|-----|-----|
| Inspector 기동 | `Week_07.md §1.5` | Lesson 05 |
| 도구 테스트 | `Week_07.md §1.5.2` | Lesson 05 |
| Claude Desktop 등록 | `Week_07.md §1.5.3` | Lesson 05 |


## §1. Inspector란? — `mcp dev` 한 줄 명령

MCP Inspector는 **브라우저 기반 테스트 도구**입니다. LLM이 없어도 도구가 정상 작동하는지 시각적으로 확인할 수 있습니다.

```bash
mcp dev tutorial_server.py
```

이 한 줄이 다음 작업을 자동으로 처리합니다:
1. `tutorial_server.py`를 stdio 서브프로세스로 기동
2. 로컬 포트(기본 6277)에 Inspector 웹 서버 시작
3. 브라우저 자동 열기 (또는 URL 표시)
4. 클라이언트(브라우저) ↔ MCP 서버 간 JSON-RPC 통신 중계

먼저 `tutorial_server.py`의 절대 경로를 확인합니다.


In [ ]:
import os

server_path = os.path.abspath("tutorial_server.py")
exists = os.path.exists("tutorial_server.py")

print(f"파일 존재 여부: {exists}")
print(f"절대 경로: {server_path}")

if not exists:
    print("\n❌ tutorial_server.py를 찾을 수 없습니다.")
    print("   → 먼저 T01_first_mcp_server.ipynb의 §8을 실행하여 파일을 생성하세요.")
else:
    print("\nOK 다음 단계로 진행하세요.")


## §2. Inspector 기동

노트북 셀에서 백그라운드 프로세스로 Inspector를 기동합니다.

> 💡 **권장**: 실제로는 별도 터미널에서 `mcp dev tutorial_server.py`를 실행하는 것이 더 편리합니다. 노트북에서는 학습 목적으로 `subprocess`로 기동합니다.

In [ ]:
import subprocess
import time
import os

# 기존 프로세스가 살아 있다면 정리
# (이미 다른 셀에서 기동한 경우 대비)

print("Inspector 기동 중... (2초 대기)")
proc = subprocess.Popen(
    ["mcp", "dev", "tutorial_server.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)
time.sleep(2)

if proc.poll() is None:
    print(f"OK Inspector PID: {proc.pid}")
    print(f"   브라우저로 열기: http://localhost:6277")
    print(f"\n   (자동으로 브라우저가 열렸을 수도 있습니다.)")
else:
    print(f"기동 실패 (returncode={proc.returncode})")
    stdout, stderr = proc.communicate(timeout=5)
    print(f"stderr: {stderr.decode('utf-8', errors='replace')[:500]}")


## §3. Inspector 화면 가이드 — 단계별

브라우저에서 `http://localhost:6277`이 열렸다면 다음 순서로 진행하세요.

### Step 1: 서버 연결
- 왼쪽 사이드바에 **"Connect"** 버튼 클릭
- 연결 성공 시 상단에 `Connected` 표시 + 서버 이름 `My First MCP Server` 노출

### Step 2: Tools 탭으로 이동
- 상단 메뉴에서 **"Tools"** 탭 클릭
- 좌측 패널에 3개 도구 표시:
  - `get_current_time`
  - `add_numbers`
  - `calculate_area`

### Step 3: 도구 선택
- 좌측 도구 목록에서 **`get_current_time`** 클릭
- 우측 패널에 입력 폼이 나타남

### Step 4: 도구 실행
- `format` 입력 칸은 비워두거나 `%Y-%m-%d %H:%M` 입력
- 우측 하단 **"Run Tool"** 또는 **"Submit"** 클릭

### Step 5: 결과 확인
- 결과창에 현재 시간 문자열 표시 (예: `2026-04-28 14:30`)
- 하단 **"History"** 탭에서 JSON-RPC 송수신 메시지 확인 가능

---

> ☑ **체크포인트 1**: Inspector가 3개 도구를 모두 보여주고 있나요?
>
> ❌ 만약 도구 목록이 비어 있다면:
> - "Connect" 버튼을 다시 클릭
> - `tutorial_server.py`에 도구가 정의되어 있는지 확인 (T01 §8)


## §4. 다른 도구도 시도

이번엔 나머지 두 도구를 테스트합니다.

### add_numbers
- 좌측에서 `add_numbers` 클릭
- `a`: `10`, `b`: `20` 입력
- Run → 결과 `30` 확인

### calculate_area
- 좌측에서 `calculate_area` 클릭
- `width`: `5`, `height`: `3`, `unit`: `m` 입력
- Run → 결과 `"15.00 m²"` 확인

> 💡 **History 탭의 가치**: 각 호출의 JSON-RPC payload가 보입니다. 이는 클라이언트(T03에서 구현)가 서버와 어떻게 통신하는지 정확히 보여주는 학습 자료입니다.


## §5. 트러블슈팅

### 포트 6277 사용 중 (`port already in use`)
이전 Inspector가 살아있는 경우:
```bash
# macOS / Linux
lsof -i :6277
kill -9 <PID>

# Windows (PowerShell)
Get-NetTCPConnection -LocalPort 6277 | Select-Object OwningProcess
Stop-Process -Id <PID> -Force
```
또는 다른 포트로 기동:
```bash
MCP_INSPECTOR_PORT=6278 mcp dev tutorial_server.py
```

### 브라우저에서 안 열림 (회사 방화벽)
- 회사망에서 localhost가 막힌 경우 → §6의 JSON-RPC 직접 통신 사용
- 또는 노트북에서 `print(...)` 한 URL을 수동으로 브라우저에 복사


## §6. JSON-RPC 직접 통신 (브라우저 막힘 대안)

브라우저가 막혔다면, 코드로 직접 stdio 통신을 해도 동일한 검증이 가능합니다. 이는 사실상 T03(클라이언트)에서 학습할 내용을 미리 맛보는 셈입니다.

In [ ]:
# JSON-RPC initialize / tools/list / tools/call 직접 송신
import asyncio
import json
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client


async def jsonrpc_inspect():
    """Inspector 없이 직접 JSON-RPC로 서버를 검증."""
    params = StdioServerParameters(
        command="python",
        args=["tutorial_server.py"],
    )

    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            # 1) initialize 핸드셰이크
            await session.initialize()
            print("OK initialize 완료")

            # 2) tools/list 호출
            tools = await session.list_tools()
            print(f"\n사용 가능한 도구 ({len(tools.tools)}개):")
            for t in tools.tools:
                print(f"  - {t.name}: {t.description}")

            # 3) tools/call 호출
            result = await session.call_tool(
                "calculate_area",
                arguments={"width": 5, "height": 3, "unit": "m"},
            )
            text = result.content[0].text if result.content else "(empty)"
            print(f"\ncalculate_area 결과: {text}")


await jsonrpc_inspect()


## §7. Claude Desktop 등록 (선택)

Inspector 테스트가 통과했다면, Claude Desktop에 서버를 등록할 수 있습니다.

```bash
mcp install tutorial_server.py
```

이 명령은 `claude_desktop_config.json`에 다음을 자동 추가합니다:

```json
{
  "mcpServers": {
    "My First MCP Server": {
      "command": "python",
      "args": ["/absolute/path/to/tutorial_server.py"]
    }
  }
}
```

Claude Desktop을 **재시작**하면 채팅창에서 자체 서버의 도구를 사용할 수 있습니다.


In [ ]:
# 실행 안내만 출력 (학생이 직접 실행 결정)
print("Claude Desktop에 등록하려면 터미널에서:")
print("  mcp install tutorial_server.py")
print()
print("등록 후 Claude Desktop 재시작 필수.")


## §8. Claude Code 등록 (선택)

Claude Code(VS Code 익스텐션 또는 CLI)에서도 자체 서버를 사용할 수 있습니다.

In [ ]:
import os

server_path = os.path.abspath("tutorial_server.py")

print("Claude Code에 등록하려면 터미널에서:")
print(f"  claude mcp add my-tutorial -- python {server_path}")
print()
print("또는 .claude/settings.json에 직접 추가:")
print('''
{
  "mcpServers": {
    "my-tutorial": {
      "command": "python",
'''.rstrip())
print(f'      "args": ["{server_path}"]')
print('''    }
  }
}
'''.lstrip())


## §9. Inspector 종료

학습이 끝나면 Inspector 프로세스를 정리합니다.

> ☑ **체크포인트 2**: 종료 후 `lsof -i :6277` 명령으로 포트가 해제되었는지 확인할 수 있습니다.

In [ ]:
# Inspector 프로세스 종료
try:
    proc.terminate()
    proc.wait(timeout=5)
    print(f"OK 종료 완료 (returncode={proc.returncode})")
except Exception as e:
    print(f"종료 중 오류: {e}")
    print("필요시 터미널에서 직접 kill하세요.")


> ☑ **체크포인트 3**: 정상 종료되었나요?
>
> 종료 후 셀을 다시 실행하면 `proc`이 이미 종료된 상태이므로 에러가 날 수 있습니다. 정상입니다.


## §10. 다음 단계

✅ T02 완료! 이제 다음을 할 수 있습니다:
- `mcp dev`로 Inspector 기동
- 브라우저 UI에서 도구 시각적 검증
- 브라우저 막힘 시 JSON-RPC 직접 통신 (T03 미리보기)
- Claude Desktop / Claude Code 등록 절차 이해

➡️ **다음 노트북**: [`T03_mcp_client_basics.ipynb`](T03_mcp_client_basics.ipynb)에서 **Python 클라이언트 코드**로 도구 호출을 자동화하고, **Claude API와 결합**하여 LLM이 자율적으로 MCP 도구를 사용하게 만듭니다.

> 📚 **추가 학습**:
> - `skilljar/S6_02_mcp_inspector.ipynb` — Skilljar 원본
> - `Week_07.md §1.5` — Inspector 심화
